In [ ]:
# ✅ Install and import necessary libraries
!pip install wandb -q

import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split
import wandb


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split
import pandas as pd

# 1. Prepare Dataset
transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.5,), (0.5,))])
dataset = datasets.MNIST(root='./data', train=True, download=True, transform=transform)

train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size
train_data, val_data = random_split(dataset, [train_size, val_size])

# 2. Define model with variable layers
class SimpleNN(nn.Module):
    def __init__(self, input_size, hidden_sizes, output_size):
        super(SimpleNN, self).__init__()
        layers = []
        in_size = input_size
        for h in hidden_sizes:
            layers.append(nn.Linear(in_size, h))
            layers.append(nn.ReLU())
            in_size = h
        layers.append(nn.Linear(in_size, output_size))
        self.model = nn.Sequential(*layers)

    def forward(self, x):
        return self.model(x)

# 3. Training function
def train_model(learning_rate, batch_size, num_layers, hidden_size, epochs=5):
    train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_data, batch_size=batch_size)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = SimpleNN(28*28, [hidden_size]*num_layers, 10).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=learning_rate)

    for epoch in range(epochs):
        model.train()
        running_loss = 0
        correct = 0
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            images = images.view(images.size(0), -1)

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()
            preds = outputs.argmax(dim=1)
            correct += (preds == labels).sum().item()

        train_acc = correct / len(train_loader.dataset)

    # Validation accuracy
    model.eval()
    correct_val = 0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            images = images.view(images.size(0), -1)
            outputs = model(images)
            preds = outputs.argmax(dim=1)
            correct_val += (preds == labels).sum().item()

    val_acc = correct_val / len(val_loader.dataset)
    return train_acc, val_acc

# 4. Run experiments
configs = [
    {'learning_rate': 0.001, 'batch_size': 32, 'num_layers': 1, 'hidden_size': 64},
    {'learning_rate': 0.001, 'batch_size': 64, 'num_layers': 1, 'hidden_size': 64},
    {'learning_rate': 0.001, 'batch_size': 64, 'num_layers': 2, 'hidden_size': 128},
    {'learning_rate': 0.0005, 'batch_size': 32, 'num_layers': 2, 'hidden_size': 64},
]

results = []

for i, cfg in enumerate(configs):
    print(f"Running config {i+1}/{len(configs)}: {cfg}")
    train_acc, val_acc = train_model(**cfg, epochs=5)
    results.append({**cfg, 'train_accuracy': train_acc, 'val_accuracy': val_acc})

# 5. Show results
df = pd.DataFrame(results)
print("\nSummary of results:")
print(df)



"""
Hyperparameters are settings defined before training a model.They are NOT learned from data.
Examples:Learning Rate (LR)Batch Size
Number of Layers,Neurons per Layer,Optimizer,Activation Functions ,Dropout Rate
Weight Decay (L2 Regularization)
You change these to improve model performance.

-Why is Hyperparameter Tuning Important?
 A bad LR can make training fail.
 Too big/small batch size affects generalization.
 Too few layers → Underfitting
 Too many layers → Overfitting
 Hyperparameters greatly influence accuracy & convergence.

PRACTICAL 8 – HYPERPARAMETER TUNING IN DEEP LEARNING
----------------------------------------------------

This program performs hyperparameter tuning on the MNIST dataset using a
Multi-Layer Perceptron (MLP) neural network. The goal is to observe how
different hyperparameters affect the training accuracy, validation accuracy,
and final test performance of the model.

WHAT THIS CODE DOES (STEP-BY-STEP):

1. Import Libraries
   - PyTorch for building and training neural networks.
   - torchvision for loading MNIST dataset.
   - pandas for creating a results table.
   - (optional) wandb for logging experiments.

2. Load and Preprocess the MNIST Dataset
   - Normalizes images and converts them into tensors.
   - Splits dataset into:
       80% → training
       20% → validation
   - Loads the test set separately for final evaluation.

3. Define a Customizable Neural Network (SimpleNN)
   - Builds an MLP with a variable number of layers.
   - Each hidden layer uses ReLU activation.
   - Output layer has 10 neurons (for digits 0–9).

4. Define the Training Function (train_model)
   - Accepts hyperparameters: learning_rate, batch_size,
     number_of_layers, hidden_size, and epochs.
   - Creates data loaders using given batch size.
   - Initializes model, loss function, and optimizer.
   - Trains the network for the given number of epochs:
        → forward pass
        → compute loss
        → backpropagation
        → optimizer step
   - Calculates training accuracy.
   - Evaluates validation accuracy after training.
   - Returns both accuracies.

5. Hyperparameter Configurations
   - Four different sets of hyperparameters are created:
        a) lr = 0.001, batch_size = 32, 1 layer, 64 neurons
        b) lr = 0.001, batch_size = 64, 1 layer, 64 neurons
        c) lr = 0.001, batch_size = 64, 2 layers, 128 neurons
        d) lr = 0.0005, batch_size = 32, 2 layers, 64 neurons

6. Run Experiments
   - Loops through each configuration.
   - Trains and evaluates the model using each hyperparameter set.
   - Stores training accuracy, validation accuracy, and model details.

7. Display Final Summary
   - Results are printed in a table format using pandas.
   - Allows easy comparison of performance between different
     hyperparameter choices.

OVERALL PURPOSE:
To study the impact of hyperparameters such as learning rate, batch size,
number of layers, and hidden neuron size on the performance of an MLP model.
This helps understand generalization, model capacity, training dynamics, and
the importance of tuning hyperparameters in deep learning.


"""

100%|██████████| 9.91M/9.91M [00:00<00:00, 17.0MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 499kB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 4.15MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 13.4MB/s]


Running config 1/4: {'learning_rate': 0.001, 'batch_size': 32, 'num_layers': 1, 'hidden_size': 64}
Running config 2/4: {'learning_rate': 0.001, 'batch_size': 64, 'num_layers': 1, 'hidden_size': 64}
Running config 3/4: {'learning_rate': 0.001, 'batch_size': 64, 'num_layers': 2, 'hidden_size': 128}
Running config 4/4: {'learning_rate': 0.0005, 'batch_size': 32, 'num_layers': 2, 'hidden_size': 64}

Summary of results:
   learning_rate  batch_size  num_layers  hidden_size  train_accuracy  \
0         0.0010          32           1           64        0.953812   
1         0.0010          64           1           64        0.956937   
2         0.0010          64           2          128        0.971313   
3         0.0005          32           2           64        0.957438   

   val_accuracy  
0      0.951833  
1      0.949833  
2      0.964000  
3      0.954167  


In [ ]:

'''
torch → main PyTorch library

torch.nn → contains neural network modules (e.g., Linear, Conv2d)

torch.optim → optimization algorithms (e.g., SGD, Adam)

torchvision.datasets → built-in datasets like MNIST, CIFAR10

torchvision.transforms → image preprocessing utilities

torch.utils.data.DataLoader → loads data in batches for training/testing

random_split → splits datasets into train/validation sets

import wandb
Imports Weights & Biases library

Used for:

Logging metrics (loss, accuracy, etc.)

Visualizing training curves

Tracking hyperparameters and model checkpoints

1. Prepare Dataset
transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.5,), (0.5,))])
dataset = datasets.MNIST(root='./data', train=True, download=True, transform=transform)


Transformations:

ToTensor() → converts images to PyTorch tensors (0–1)

Normalize((0.5,), (0.5,)) → scales data to roughly [-1, 1]

datasets.MNIST(...) → loads MNIST training dataset

train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size
train_data, val_data = random_split(dataset, [train_size, val_size])


Splits dataset into 80% train and 20% validation

2. Define model with variable layers
class SimpleNN(nn.Module):
    def __init__(self, input_size, hidden_sizes, output_size):
        super(SimpleNN, self).__init__()
        layers = []
        in_size = input_size
        for h in hidden_sizes:
            layers.append(nn.Linear(in_size, h))
            layers.append(nn.ReLU())
            in_size = h
        layers.append(nn.Linear(in_size, output_size))
        self.model = nn.Sequential(*layers)

    def forward(self, x):
        return self.model(x)


Builds a fully connected feedforward network (MLP) dynamically:

input_size → flattened image size (28×28=784)

hidden_sizes → list of hidden layer sizes

output_size → number of classes (10 for MNIST)

Each hidden layer uses ReLU activation

Final layer has no activation because CrossEntropyLoss expects logits

3. Training function
def train_model(learning_rate, batch_size, num_layers, hidden_size, epochs=5):


Function allows hyperparameter experiments:

learning_rate, batch_size, num_layers, hidden_size

train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_data, batch_size=batch_size)


Creates train and validation data loaders

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = SimpleNN(28*28, [hidden_size]*num_layers, 10).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=learning_rate)


Moves model to GPU if available

Creates SimpleNN dynamically based on number of layers & hidden size

Uses Adam optimizer with specified learning rate

Loss: CrossEntropyLoss for multi-class classification

for epoch in range(epochs):
    model.train()
    running_loss = 0
    correct = 0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        images = images.view(images.size(0), -1)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        preds = outputs.argmax(dim=1)
        correct += (preds == labels).sum().item()

    train_acc = correct / len(train_loader.dataset)


Standard training loop:

Flatten images

Forward pass

Compute loss

Backward pass

Update weights

Compute training accuracy

model.eval()
correct_val = 0
with torch.no_grad():
    for images, labels in val_loader:
        images, labels = images.to(device), labels.to(device)
        images = images.view(images.size(0), -1)
        outputs = model(images)
        preds = outputs.argmax(dim=1)
        correct_val += (preds == labels).sum().item()

val_acc = correct_val / len(val_loader.dataset)
return train_acc, val_acc


Validation loop:

Disables gradient computation (torch.no_grad())

Computes validation accuracy

4. Run experiments
configs = [
    {'learning_rate': 0.001, 'batch_size': 32, 'num_layers': 1, 'hidden_size': 64},
    {'learning_rate': 0.001, 'batch_size': 64, 'num_layers': 1, 'hidden_size': 64},
    {'learning_rate': 0.001, 'batch_size': 64, 'num_layers': 2, 'hidden_size': 128},
    {'learning_rate': 0.0005, 'batch_size': 32, 'num_layers': 2, 'hidden_size': 64},
]


List of hyperparameter combinations to test

results = []

for i, cfg in enumerate(configs):
    print(f"Running config {i+1}/{len(configs)}: {cfg}")
    train_acc, val_acc = train_model(**cfg, epochs=5)
    results.append({**cfg, 'train_accuracy': train_acc, 'val_accuracy': val_acc})


Loops over all configurations

Calls train_model

Stores train & validation accuracies with the hyperparameters

5. Show results
df = pd.DataFrame(results)
print("\nSummary of results:")
print(df)


Converts results into a pandas DataFrame for easy visualization

Prints summary table showing which hyperparameters performed best